# 数据集切分与泄漏：按样本随机切为什么会虚高

**面试问题：训练、验证、测试集应该怎样切，如何识别用户级、时间级和目标泄漏？**

## 回答主线

1. 切分单位必须与上线泛化单位一致：要泛化到新用户就按用户分组，而不是随机切工单行。
2. 同一用户的重复样本跨集合会让模型记住身份，离线分数虚高。
3. 时间预测还必须保证训练时间早于验证和测试，不能让未来特征进入过去。
4. 预处理器、词表、归一化统计和特征选择都只能在训练集拟合。
5. 目标发生后才产生的状态字段即使在测试集也可见，仍属于 target leakage。
6. 合格报告要打印组重叠、时间范围、标签分布和泄漏探针。

## 真实案例

六名客户各有三张客服工单，客户是否流失在同一客户内固定。我们构造一个“记住 customer_id”的最小模型：随机按行切分时测试客户都在训练中，准确率虚高；按客户分组后，新客户无法被记忆。随后加入结单后才写入的 `outcome_hint`，复现即便分组切分也能作弊。使用可读的离线教学数据解释机制，指标不能外推为线上收益。

### 输入预览：六个客户、十八张重复工单

In [1]:
records = []  # 收集十八张具有客户分组和时间的工单。
customer_labels = {"C1": 1, "C2": 0, "C3": 1, "C4": 0, "C5": 1, "C6": 0}  # 定义每个客户最终是否流失。
for customer_index, (customer_id, churned) in enumerate(customer_labels.items(), start=1):  # 逐客户生成三次历史交互。
    for visit in range(3):  # 同一客户有三张高度相关工单。
        records.append({"customer": customer_id, "day": customer_index * 10 + visit, "complaint_count": visit + customer_index % 2, "channel": "app" if visit % 2 == 0 else "phone", "churned": churned})  # 保存业务字段和标签。
print("前九条记录：")  # 输出数据预览标题。
print("customer day complaints channel churned")  # 输出表头。
for record in records[:9]:  # 展示三个客户的重复记录。
    print(f"{record['customer']:<8} {record['day']:>3} {record['complaint_count']:>10} {record['channel']:<7} {record['churned']}")  # 展示组内标签相同。
print(f"总行数={len(records)}，客户数={len(customer_labels)}，正例行数={sum(record['churned'] for record in records)}")  # 汇总规模与标签。

前九条记录：
customer day complaints channel churned
C1        10          1 app     1
C1        11          2 phone   1
C1        12          3 app     1
C2        20          0 app     0
C2        21          1 phone   0
C2        22          2 app     0
C3        30          1 app     1
C3        31          2 phone   1
C3        32          3 app     1
总行数=18，客户数=6，正例行数=9


## Baseline 基线：按行交错切分并记住 Customer ID

In [2]:
random_like_test_indices = {2, 5, 8, 11, 14, 17}  # 模拟每个客户随机抽一行进入测试集。
row_train = [record for index, record in enumerate(records) if index not in random_like_test_indices]  # 其余十二行作为训练集。
row_test = [record for index, record in enumerate(records) if index in random_like_test_indices]  # 每个客户一行作为测试集。

def fit_customer_lookup(train_rows):  # 训练一个暴露组泄漏的客户记忆模型。
    labels_by_customer = {}  # 收集训练集中每个客户标签。
    for row in train_rows:  # 遍历训练行。
        labels_by_customer.setdefault(row["customer"], []).append(row["churned"])  # 保存同客户历史标签。
    mapping = {customer: int(sum(labels) >= len(labels) / 2) for customer, labels in labels_by_customer.items()}  # 用多数标签记住每个客户。
    fallback = int(sum(row["churned"] for row in train_rows) >= len(train_rows) / 2)  # 对未知客户使用训练集多数类。
    return mapping, fallback  # 返回身份映射和未知客户回退。

def evaluate_lookup(train_rows, test_rows):  # 拟合并逐样本评估客户记忆模型。
    mapping, fallback = fit_customer_lookup(train_rows)  # 只使用训练集建立映射。
    rows = []  # 收集测试预测。
    for row in test_rows:  # 遍历测试样本。
        prediction = mapping.get(row["customer"], fallback)  # 已见客户直接取记忆标签。
        rows.append({"customer": row["customer"], "seen": row["customer"] in mapping, "actual": row["churned"], "prediction": prediction, "correct": prediction == row["churned"]})  # 保存泄漏证据。
    return rows  # 返回逐客户结果。

row_results = evaluate_lookup(row_train, row_test)  # 运行按行切分基线。
row_overlap = sorted({row["customer"] for row in row_train}.intersection({row["customer"] for row in row_test}))  # 计算训练测试客户重叠。
print("按行切分测试结果：")  # 输出虚高结果。
for row in row_results:  # 逐测试行展示是否见过身份。
    print(row)  # 让全部 seen=True 直接可见。
print(f"客户重叠={row_overlap}，准确率={sum(row['correct'] for row in row_results) / len(row_results):.0%}")  # 汇总泄漏程度。

按行切分测试结果：
{'customer': 'C1', 'seen': True, 'actual': 1, 'prediction': 1, 'correct': True}
{'customer': 'C2', 'seen': True, 'actual': 0, 'prediction': 0, 'correct': True}
{'customer': 'C3', 'seen': True, 'actual': 1, 'prediction': 1, 'correct': True}
{'customer': 'C4', 'seen': True, 'actual': 0, 'prediction': 0, 'correct': True}
{'customer': 'C5', 'seen': True, 'actual': 1, 'prediction': 1, 'correct': True}
{'customer': 'C6', 'seen': True, 'actual': 0, 'prediction': 0, 'correct': True}
客户重叠=['C1', 'C2', 'C3', 'C4', 'C5', 'C6']，准确率=100%


### 核心实现：按客户 Group Split 并检查时间范围

In [3]:
train_customers = {"C1", "C2", "C3", "C4"}  # 将前四个客户完整放入训练集。
test_customers = {"C5", "C6"}  # 将两个从未出现的客户完整放入测试集。
group_train = [record for record in records if record["customer"] in train_customers]  # 生成无跨组训练集。
group_test = [record for record in records if record["customer"] in test_customers]  # 生成无跨组测试集。
group_results = evaluate_lookup(group_train, group_test)  # 在新客户上评估身份记忆模型。
group_overlap = sorted({row["customer"] for row in group_train}.intersection({row["customer"] for row in group_test}))  # 检查客户交集应为空。
train_time = (min(row["day"] for row in group_train), max(row["day"] for row in group_train))  # 统计训练时间范围。
test_time = (min(row["day"] for row in group_test), max(row["day"] for row in group_test))  # 统计测试时间范围。
print("Group Split 测试结果：")  # 输出真实新客户结果。
for row in group_results:  # 逐测试样本展示身份未知。
    print(row)  # 显示 seen=False 与回退预测。
print(f"客户重叠={group_overlap}，训练日期={train_time}，测试日期={test_time}，准确率={sum(row['correct'] for row in group_results) / len(group_results):.0%}")  # 展示无泄漏切分更诚实。

Group Split 测试结果：
{'customer': 'C5', 'seen': False, 'actual': 1, 'prediction': 1, 'correct': True}
{'customer': 'C5', 'seen': False, 'actual': 1, 'prediction': 1, 'correct': True}
{'customer': 'C5', 'seen': False, 'actual': 1, 'prediction': 1, 'correct': True}
{'customer': 'C6', 'seen': False, 'actual': 0, 'prediction': 1, 'correct': False}
{'customer': 'C6', 'seen': False, 'actual': 0, 'prediction': 1, 'correct': False}
{'customer': 'C6', 'seen': False, 'actual': 0, 'prediction': 1, 'correct': False}
客户重叠=[]，训练日期=(10, 42)，测试日期=(50, 62)，准确率=50%


## 结果解读：同一个模型在两种切分下的巨大差异

In [4]:
row_accuracy = sum(row["correct"] for row in row_results) / len(row_results)  # 计算按行切分准确率。
group_accuracy = sum(row["correct"] for row in group_results) / len(group_results)  # 计算按客户切分准确率。
print("切分方案        测试行  测试客户  客户重叠  身份已见率  accuracy")  # 输出对照表头。
print(f"随机按行        {len(row_test):>6} {len({row['customer'] for row in row_test}):>8} {len(row_overlap):>8} {sum(row['seen'] for row in row_results) / len(row_results):>10.0%} {row_accuracy:>9.0%}")  # 展示泄漏方案。
print(f"按客户+时间     {len(group_test):>6} {len(test_customers):>8} {len(group_overlap):>8} {sum(row['seen'] for row in group_results) / len(group_results):>10.0%} {group_accuracy:>9.0%}")  # 展示上线一致方案。
print("解读：100% 不是模型学会流失规律，而是训练集中已出现每个测试 customer_id；新客户结果才回答真实泛化问题。")  # 明确指标差异原因。

切分方案        测试行  测试客户  客户重叠  身份已见率  accuracy
随机按行             6        6        6       100%      100%
按客户+时间          6        2        0         0%       50%
解读：100% 不是模型学会流失规律，而是训练集中已出现每个测试 customer_id；新客户结果才回答真实泛化问题。


## 失败案例：结单后 Outcome Hint 让 Group Split 也虚高

In [5]:
leaky_group_test = [{**row, "outcome_hint": "will_churn" if row["churned"] else "will_stay"} for row in group_test]  # 加入标签发生后才写入的结单字段。
def outcome_hint_predict(rows):  # 模拟模型利用结果字段作弊。
    return [{"customer": row["customer"], "prediction": int(row["outcome_hint"] == "will_churn"), "actual": row["churned"]} for row in rows]  # 直接从提示恢复标签。

leaky_predictions = outcome_hint_predict(leaky_group_test)  # 在无客户重叠测试集上运行泄漏特征。
leaky_accuracy = sum(row["prediction"] == row["actual"] for row in leaky_predictions) / len(leaky_predictions)  # 计算作弊准确率。
feature_availability = {"customer": "prediction_time", "complaint_count": "prediction_time", "channel": "prediction_time", "outcome_hint": "after_outcome"}  # 建立特征可用时间清单。
allowed_features = [feature for feature, available_at in feature_availability.items() if available_at == "prediction_time"]  # 只保留预测时可用字段。
print("带 outcome_hint 的 Group Test：", leaky_predictions)  # 展示分组切分仍可被目标泄漏破坏。
print(f"作弊准确率={leaky_accuracy:.0%}，特征可用时间={feature_availability}，门禁后特征={allowed_features}")  # 展示修正策略。
print("生产边界：真实管道还要冻结切分 manifest、对用户/设备/文档去重、按事件时间 join，并确保所有预处理器只 fit train。")  # 说明工程检查。

带 outcome_hint 的 Group Test： [{'customer': 'C5', 'prediction': 1, 'actual': 1}, {'customer': 'C5', 'prediction': 1, 'actual': 1}, {'customer': 'C5', 'prediction': 1, 'actual': 1}, {'customer': 'C6', 'prediction': 0, 'actual': 0}, {'customer': 'C6', 'prediction': 0, 'actual': 0}, {'customer': 'C6', 'prediction': 0, 'actual': 0}]
作弊准确率=100%，特征可用时间={'customer': 'prediction_time', 'complaint_count': 'prediction_time', 'channel': 'prediction_time', 'outcome_hint': 'after_outcome'}，门禁后特征=['customer', 'complaint_count', 'channel']
生产边界：真实管道还要冻结切分 manifest、对用户/设备/文档去重、按事件时间 join，并确保所有预处理器只 fit train。


## 回归测试：最后只保护组隔离、时间与目标泄漏探针

In [6]:
assert row_overlap == ["C1", "C2", "C3", "C4", "C5", "C6"] and row_accuracy == 1.0  # 验证按行方案让所有测试客户泄漏。
assert group_overlap == [] and all(not row["seen"] for row in group_results)  # 验证 Group Split 测试客户从未出现在训练。
assert train_time[1] < test_time[0]  # 验证本例训练时间严格早于测试时间。
assert group_accuracy == 0.5 and group_accuracy < row_accuracy  # 验证诚实新客户指标显著低于记忆指标。
assert leaky_accuracy == 1.0 and "outcome_hint" not in allowed_features  # 验证目标泄漏反例和特征时间门禁。
print("回归测试通过：客户重叠、组隔离、时间顺序、诚实指标和 Outcome 泄漏均成立。")  # 用少量断言总结切分合同。

回归测试通过：客户重叠、组隔离、时间顺序、诚实指标和 Outcome 泄漏均成立。
